# WildChat annotation compiler

Loads the immutable Arrow shards written by `annotate.ipynb` under
`../data/wildchat-annotations/shards/` and concatenates them into **one dataset**
ready for downstream use.

- **Read-only**: nothing in the shard dir is written, renamed or deleted
  (unlike `AnnotationStore`, which sweeps leftover `.tmp` dirs on init),
  so this is safe to run even while an annotation run is in progress -
  it just sees the shards that exist at the moment it runs. Re-run to refresh.
- **No secrets / network**: this never talks to the annotator, so no `.env`,
  no OpenAI/HF tokens, no WildChat download.
- **Schema-safe**: refuses to merge shards whose features drifted (e.g. a
  `THEMES` change mid-collection) instead of failing obscurely later.

Recipes for downstream use are at the bottom.

In [ ]:
from pathlib import Path

from datasets import load_from_disk, concatenate_datasets

# resolve the project root no matter where the kernel was started from
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
SHARD_DIR = PROJECT_ROOT / "data" / "wildchat-annotations" / "shards"

if not SHARD_DIR.is_dir():
    raise FileNotFoundError(f"no shard dir at {SHARD_DIR} - run annotate.ipynb first")

# a finished shard is a directory with state.json that is not a partial .tmp write
shard_dirs = [d for d in sorted(SHARD_DIR.iterdir())
              if d.is_dir() and not d.name.endswith(".tmp") and (d / "state.json").exists()]
if not shard_dirs:
    raise RuntimeError(f"{SHARD_DIR} exists but holds no finished shards yet - run annotate.ipynb first")

print(f"{len(shard_dirs)} finished shards under {SHARD_DIR.relative_to(PROJECT_ROOT)}")

In [ ]:
# load every shard, guard against schema drift, merge into one dataset
shards = [load_from_disk(str(d)) for d in shard_dirs]

# concatenate_datasets requires identical schemas; shards written by older code
# (different THEMES list, renamed columns, ...) would fail here or poison the merge
base_features = shards[0].features
drifted = [d.name for ds, d in zip(shards, shard_dirs) if ds.features != base_features]
if drifted:
    raise ValueError(
        f"schema drift in {len(drifted)}/{len(shards)} shards "
        f"(first offender: {drifted[0]}) - features differ from {shard_dirs[0].name}. "
        "Likely THEMES or the record layout changed after those shards were written."
    )

annotated = concatenate_datasets(shards)
errored = annotated.filter(lambda e: e != "", input_columns="error").num_rows
print(f"{annotated.num_rows} messages from {len(shards)} shards "
    f"({errored} rows carry at least one failed request)")
annotated

In [ ]:
# recipes for downstream use
# theme columns = everything that is not one of the fixed record columns
FIXED_COLUMNS = ["conversation_id", "message_idx", "role", "content_hash",
                 "content", "model", "rationale", "error", "timestamp"]
THEMES = [c for c in annotated.column_names if c not in FIXED_COLUMNS]

clean = annotated.filter(lambda err: err == "", input_columns="error")

# per-role mean scores (role-inapplicable themes are None and skipped by pandas)
by_role = clean.to_pandas().groupby("role")[THEMES].mean()
by_role

# reconstruct one conversation with annotations aligned to message order:
# conv_id = clean[0]["conversation_id"]
# conv = clean.filter(lambda cid: cid == conv_id, input_columns="conversation_id").sort("message_idx")

# persist one merged copy for consumers that don't want to touch the shard dir:
# clean.save_to_disk(PROJECT_ROOT / "data" / "wildchat-annotations-compiled")